In [ ]:
import re
import math
import os
import pandas as pd

# =============================================================================
# 1. データパーサー (パラメータ読み込み)
# =============================================================================
def parse_dat_file(dat_filepath):
    # デフォルト値 (model.txtの挙動に合わせて設定)
    params = {
        'scaleLength': 635.0,
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 24, 'p_max': 24,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.5, 2.0, 3.0],
        'natural_min_spread_mm': [20.0, 40.0, 60.0],
        'availableTime': [],
        # Weights
        'd_PC': 1.5, 'd_SC': 1.0, 'd_ST': 1.0, 'd_DH': 1.0, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2
    }

    print(f"DEBUG: Parsing .dat file: {dat_filepath}")
    if not os.path.exists(dat_filepath):
        print("WARNING: File not found. Using defaults.")
        return params

    try:
        content = ""
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(dat_filepath, 'r', encoding=enc) as f:
                    content = f.read()
                break
            except UnicodeDecodeError: continue

        # Helper to extract values
        def get_val(name, cast_func):
            # Scalar
            pat = re.compile(rf'{name}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: return cast_func(m.group(1))
            return None

        def get_arr(name):
            # Array [v1, v2, ...]
            pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
            m = pat.search(content)
            if m:
                txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
                return [float(v) for v in txt.split(',') if v.strip()]
            return None

        # Update params if found in file
        for k in params.keys():
            if isinstance(params[k], list):
                val = get_arr(k)
                if val: params[k] = val
            else:
                val = get_val(k, float)
                if val is not None: params[k] = val
        
        # Casting integers
        params['s_max'] = int(params['s_max'])
        params['f_max'] = int(params['f_max'])
        
        return params
    except Exception as e:
        print(f"ERROR: Parsing failed ({e}). Using defaults.")
        return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath): return []
    
    try:
        lines = []
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(solution_filepath, 'r', encoding=enc) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError: continue

        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue
            if not header_found:
                if "Note" in line and "String" in line: header_found = True
                continue
            if line.startswith('---'): break
            
            parts = re.split(r'\s+', line)
            if parts[0] == "Note": continue
            if len(parts) >= 5:
                try:
                    note = {
                        'note': parts[0],
                        'string': int(parts[1]),
                        'fret': int(parts[2]),
                        'hand': int(parts[3]),
                        'finger': 0 if parts[4] == 'x' else int(parts[4])
                    }
                    notes_data.append(note)
                except ValueError: continue
        return notes_data
    except: return []

# =============================================================================
# 2. モデル計算ロジック (model.txt 再現)
# =============================================================================
class GuitarModelReproducer:
    def __init__(self, params):
        self.p = params
        
        # 1-based indexing adjustment
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']
        self.fingerDiff = self.p['fingerDifficulty'] # 0-based in python
        self.knuckleDist = [0.0, 0.0] + self.p['natural_min_spread_mm']
        
        # Constants from model.txt
        self.limitRatios = [0.0, 1.0, 1.05, 1.05, 1.2]
        self.baseFactors = [0.0, 0.9, 1.0, 1.0, 1.0]
        self.stringTension = {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.2, 5: 1.4, 6: 1.6}
        
        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        self.tanTheta = math.tan(15.0 * self.deg_to_rad)
        
        # Fret Positions
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]

    def calculate_costs(self, hand_pos, finger_idx, string, fret):
        """
        model.txt の Section B & C を再現
        戻り値: (Stretch_Cost, Physics_Cost)
        """
        # 0. 論理制約
        if fret < hand_pos: return (100000.0, 0.0)

        # --- 1. 幾何計算 (Stretch) ---
        # model.txt Source 21: pivotY = 0.0
        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0 

        # Source 24: Rotation
        maxRotation = 45.0
        thetaDeg = maxRotation * (string - 6.0) / 5.0
        thetaRad = thetaDeg * self.deg_to_rad

        d = self.knuckleDist[finger_idx]
        kX = pivotX + (d * math.cos(thetaRad))
        kY = pivotY + (d * math.sin(thetaRad))

        # Source 26: Target
        tX = self.fret_pos[fret]
        tY = (string - 1) * self.p['stringSpacing']

        vX = tX - kX; vY = tY - kY
        dist = math.sqrt(vX*vX + vY*vY)

        # Source 41: Angle
        # Normal vector: (-sin, cos)
        nX = -math.sin(thetaRad)
        nY = math.cos(thetaRad)
        dotProd = vX * nX + vY * nY
        cosDelta = dotProd / dist if dist > 0.001 else 1.0
        cosDelta = max(-1.0, min(1.0, cosDelta))
        deltaDeg = math.acos(cosDelta) * self.rad_to_deg

        # --- 2. 有効長補正 (Neck Joint) ---
        # Source 29-31
        neckFactor = 1.0
        if fret >= 15:
            ratioHigh = (fret - 15.0) / (self.p['f_max'] - 15.0)
            if ratioHigh > 1.0: ratioHigh = 1.0
            neckFactor = 1.0 - (0.15 * ratioHigh)
        
        eff_len = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] * neckFactor
        limit = self.limitRatios[finger_idx]

        # Impossible Check
        if eff_len > 0 and dist > eff_len * limit:
            return (100000.0, 0.0)

        # --- 3. Stretch Cost Calculation ---
        distCost = 0.0
        if eff_len > 0:
            ratio = dist / eff_len
            if ratio < 0.3: distCost = (0.3 - ratio) * 5.0
            elif ratio > 0.7:
                distCost = (ratio - 0.7) * 10.0
                if ratio > 1.0: distCost += (ratio - 1.0) * 20.0

        baseMaxAngle = 30.0 if finger_idx == 4 else 25.0
        maxAngle = baseMaxAngle
        if fret >= 12:
            maxAngle = max(10.0, baseMaxAngle - (fret - 12) * 0.5)
        
        angleCost = (deltaDeg - maxAngle) * 0.5 if deltaDeg > maxAngle else 0.0

        diff = self.fingerDiff[finger_idx - 1]
        # Source 47: (dist + angle) * diff + (diff - 1.0)
        st_val = (distCost + angleCost) * diff + (diff - 1.0)

        # --- 4. Physics Cost Calculation (Grimes Model) ---
        # Source 48-52
        a_mm = 0.5 + (2.0 * fret / 12.0) # Action
        l_mm = self.p['scaleLength'] - self.fret_pos[fret] # Vibrating Length
        tau = self.stringTension.get(string, 1.0)
        
        if l_mm <= 0: forceIndex = 0
        else: forceIndex = (2.0 * tau * a_mm) / (l_mm * self.tanTheta)
        
        # Source 52: Scaling & Difficulty
        ph_val = (forceIndex * 200.0) * diff

        return (st_val, ph_val)

    def calculate_total(self, notes_data):
        total = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
        details = []
        
        num_notes = len(notes_data)
        avail_time = self.p['availableTime'] if self.p['availableTime'] else [30.0] * (num_notes + 10)
        
        prev_idx_map = {}
        acc_time = {}

        # Time Calculation
        for i in range(num_notes):
            is_rest = (notes_data[i]['note'] == 'R')
            if not is_rest:
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                if p >= 0:
                    prev_idx_map[i] = p
                    acc_time[i] = t_sum + avail_time[p]
                else:
                    prev_idx_map[i] = i
                    acc_time[i] = 10000.0
            else:
                acc_time[i] = 0.0
                prev_idx_map[i] = -1

        # Main Loop
        for i in range(num_notes):
            curr = notes_data[i]
            row = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
            
            if curr['note'] != 'R':
                # Time Penalty
                time_factor = min(1.0, 50.0 / (acc_time[i] + 0.01))
                
                # NO (Open String)
                if curr['fret'] == 0: row['NO'] = 1.0

                # ST & DH (Physics)
                if curr['finger'] in [1,2,3,4] and curr['fret'] > 0:
                    st_raw, ph_raw = self.calculate_costs(
                        curr['hand'], curr['finger'], curr['string'], curr['fret']
                    )
                    if st_raw >= 50000: st_raw = 100000.0 # Cap for display
                    row['ST'] = st_raw
                    row['DH'] = ph_raw

                # PC & SC
                prev_i = prev_idx_map.get(i, i)
                if prev_i != i and prev_i != -1:
                    prev = notes_data[prev_i]
                    row['PC'] = abs(self.fret_pos[prev['hand']] - self.fret_pos[curr['hand']])
                    row['SC'] = self.p['stringSpacing'] * abs(prev['string'] - curr['string'])
                    
                    # Joint Savings
                    if (curr['finger'] == 1 and prev['finger'] == 1 and 
                        curr['fret'] == prev['fret'] and abs(curr['string'] - prev['string']) >= 1):
                        row['SC'] -= (self.p['stringSpacing'] * 0.9)

                # Apply Weights (Alpha * d_XX)
                # model.txt Source 79
                w_pc = row['PC'] * time_factor * self.p['d_PC'] * self.p['alpha_PC']
                w_sc = row['SC'] * time_factor * self.p['d_SC'] * self.p['alpha_SC']
                w_st = row['ST'] * self.p['d_ST'] * self.p['alpha_ST'] # Use d_ST
                w_dh = row['DH'] * self.p['d_DH'] * self.p['alpha_DH'] # Use d_DH for Physics
                w_no = row['NO'] * self.p['d_NO'] * self.p['alpha_NO']

                total['PC'] += w_pc
                total['SC'] += w_sc
                total['ST'] += w_st
                total['DH'] += w_dh
                total['NO'] += w_no
                
                sum_weighted = w_pc + w_sc + w_st + w_dh + w_no
                details.append({
                    'Index': i+1, 'Note': curr['note'],
                    'PC_w': w_pc, 'SC_w': w_sc, 'ST_w': w_st, 'PH_w': w_dh,
                    'ST_raw': row['ST'], 'PH_raw': row['DH'],
                    'Total': sum_weighted
                })

        grand_total = sum(total.values())
        return total, grand_total, pd.DataFrame(details)

# =============================================================================
# 3. メイン実行
# =============================================================================
if __name__ == "__main__":
    # Settings
    dat_file = "complex_solo.dat"
    sol_file = "complex_opt.txt"
    
    # Paths
    if not os.path.exists(dat_file) and os.path.exists(os.path.join("datas", dat_file)):
        dat_file = os.path.join("datas", dat_file)
    if not os.path.exists(sol_file) and os.path.exists(os.path.join("results", sol_file)):
        sol_file = os.path.join("results", sol_file)

    # Run
    params = parse_dat_file(dat_file)
    model = GuitarModelReproducer(params)
    
    print(f"\nReading solution: {sol_file}")
    notes = parse_solution_file(sol_file)
    
    if notes:
        bd, total, df = model.calculate_total(notes)
        
        print("\n" + "="*60)
        print(" --- Cost Breakdown (Reproduced from model.txt) ---")
        print(f"  Position Change (PC): {bd['PC']:.4f}")
        print(f"  String Change   (SC): {bd['SC']:.4f}")
        print(f"  Geo Stretch     (ST): {bd['ST']:.4f} (d_ST={params['d_ST']})")
        print(f"  Phys Action     (DH): {bd['DH']:.4f} (d_DH={params['d_DH']})")
        print(f"  Open String     (NO): {bd['NO']:.4f}")
        print("-" * 40)
        print(f"  Total Cost (Sum)    : {total:.4f}")
        print("="*60)
        
        if not df.empty:
            print("\n[Details - Top 20]")
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            print(df.to_string(index=False, float_format="%.3f"))
    else:
        print("No valid notes found to calculate.")

DEBUG: Parsing .dat file: datas/complex_solo.dat

Reading solution: complex_opt.txt

 --- Cost Breakdown (Reproduced from model.txt) ---
  Position Change (PC): 208.9393
  String Change   (SC): 64.1255
  Geo Stretch     (ST): 48.3620 (d_ST=0.5)
  Phys Action     (DH): 126.8295 (d_DH=0.3)
  Open String     (NO): 0.0000
----------------------------------------
  Total Cost (Sum)    : 448.2563

[Details - Top 20]
 Index Note   PC_w  SC_w  ST_w  PH_w  ST_raw  PH_raw  Total
     1   A4  0.000 0.000 0.614 0.545   6.143   9.076  1.159
     2   A4  0.000 0.000 0.614 0.545   6.143   9.076  1.159
     3   A4  0.000 0.000 0.614 0.545   6.143   9.076  1.159
     4   G4  0.000 1.728 0.040 0.987   0.400  16.456  2.755
     5   F4  0.000 0.000 0.100 0.545   1.000   9.076  0.645
     6   E4  8.477 0.000 0.100 0.474   1.000   7.907  9.051
     7   F4  0.000 0.000 0.020 0.653   0.200  10.891  0.673
     8   E4  0.000 0.000 0.100 0.474   1.000   7.907  0.574
     9   D4  8.477 1.728 0.040 1.185   0.400  